# 01 Chunk With BGE-M3

Generate fixed, parent-child, and BGE-M3 semantic chunks in artifact-only mode.

This notebook supports the shared partition contract. The recommended default is to split Kaggle versions by strategy.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)

CORPUS_DIR = Path(f'/kaggle/input/{CORPUS_SLUG}/benchmark/tuvi_golden_dataset')
SCRIPTS_DIR = Path(f'/kaggle/input/{SCRIPTS_SLUG}')
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES] if PARTITION_MODE == 'strategy' else list(ALL_STRATEGIES)
ACTIVE_SOURCES = [item for item in SOURCE_IDS if item in SELECTED_SOURCES] if PARTITION_MODE == 'source' else list(SOURCE_IDS)
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')
if not ACTIVE_SOURCES:
    raise ValueError('No active sources selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
REPORTS = OUTPUT_ROOT / 'reports'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

INPUTS = [str(CORPUS_DIR / 'corpus' / source_id) for source_id in ACTIVE_SOURCES]
print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'active_sources': ACTIVE_SOURCES}, ensure_ascii=False, indent=2))

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    cmd = [
        sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'chunk_text.py'),
        '--input', *INPUTS,
        '--chunking-strategy', strategy,
        '--output', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--summary-output', str(REPORTS / f'{strategy}_chunk_summary.json'),
    ]
    if strategy == 'chunk_semantic_embedding_bge_m3':
        cmd += [
            '--semantic-report-output', str(REPORTS / f'{strategy}_semantic_similarity_report.json'),
            '--embedding-backend', 'local',
            '--local-embedding-model', 'BAAI/bge-m3',
            '--local-embedding-batch-size', '16',
        ]
    print('Running strategy =', strategy)
    subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

skipped_strategies = [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES]
print('Skipped strategies =', skipped_strategies)

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    summary_path = REPORTS / f'{strategy}_chunk_summary.json'
    print(strategy, 'summary exists =', summary_path.exists(), 'path =', summary_path)
print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])